In [23]:
import ssl
import pandas as pd
import numpy as np
from urllib.error import URLError
from urllib.request import urlopen
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
DATA_URL = "https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv"

try:
    df = pd.read_csv(DATA_URL, sep=";")
except URLError:
    ssl_context = ssl._create_unverified_context()
    with urlopen(DATA_URL, context=ssl_context) as response:
        df = pd.read_csv(response, sep=";")

In [25]:
df["Income"] = df["Income"].fillna(round(df["Income"].mean(), 1))

In [19]:
df["Age"] = 2014 - df["Year_Birth"]
df["Total_Spending"] = df[["MntWines", "MntFruits", "MntMeatProducts",
  "MntFishProducts", "MntSweetProducts", "MntGoldProds"]].sum(axis=1)
df["Total_Purchases"] = df[["NumDealsPurchases", "NumWebPurchases",
 "NumCatalogPurchases", "NumStorePurchases"]].sum(axis=1)
df["Children"] = df["Kidhome"] + df["Teenhome"]

features = ["Income", "Age", "Total_Spending", "Total_Purchases", "Recency", "Children"]
X = df[features]


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) 

In [21]:
sil_scores = {}
for k in range(2, 10):
 km = KMeans(n_clusters=k, random_state=52, n_init=10).fit(X_scaled)
 sil_scores[k] = silhouette_score(X_scaled, km.labels_)

print("Silhouette scores:", {k:round(v, 3) for k, v in sil_scores.items()})
best_k = max(sil_scores, key=sil_scores.get)
print(f"Best k by silhouette score: {best_k}")


Silhouette scores: {2: 0.315, 3: 0.239, 4: 0.211, 5: 0.215, 6: 0.209, 7: 0.211, 8: 0.216, 9: 0.221}
Best k by silhouette score: 2


In [22]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

print(df["Cluster"].value_counts().sort_index())

Cluster
0    1225
1    1015
Name: count, dtype: int64
